# hashing.ipynb
## Creator: kccaterworld
## Contributors: kccaterworld
### Status: Unfinished, Stable (Techically)

In [19]:
def checkBin(bins, length=32):
    for number in bins:
        if len(number) != length:
            raise ValueError(f"Binary string must be {length} bits long")
        if not all(bit in '01' for bit in number):
            raise ValueError("Input must be a binary string")
        if type(number) != str:
            raise TypeError("Input must be a string")

def binRot(bin, n):
    checkBin((bin,))
    if not (0 <= n <= 32):
        raise ValueError("Rotation amount must be between 0 and 32")
    return bin[n:] + bin[:n]

def binPadder(bin, length):
    if length < len(bin):
        raise ValueError("Length must be greater than or equal to the binary string length")
    while len(bin) < length:
        bin = "0" + bin
    return bin


In [ ]:
def bAnd(binA, binB):
    result = ""
    for i in range(len(binA)):
        if binA[i] == "0" or binB[i] == "0":
            result += "0"
            continue
        if binA[i] == "1" and binB[i] == "1":
            result += "1"
            continue
    return result

def bOr(binA, binB):
    result = ""
    for i in range(len(binA)):
        if binA[i] == "1" or binB[i] == "1":
            result += "1"
            continue
        if binA[i] == "0" and binB[i] == "0":
            result += "0"
            continue
    return result

def bXor(binA, binB):
    result = ""
    for i in range(len(binA)):
        if binA[i] != binB[i]:
            result += "1"
            continue
        if binA[i] == binB[i]:
            result += "0"
            continue
    return result

def bNot(bin):
    result = ""
    for digit in bin:
        result += f"{"1" if digit == "0" else "0"}"
        continue
    return result

def bAdd(binA, binB):
    intA = int(binA, 2)
    intB = int(binB, 2)
    return binPadder(str(bin(intA + intB)[2:]), 32)

In [122]:
def concat(lists):
    result = []
    for list in lists:
        result += list
    return result

def prettyPrint(text):
    returned = ""
    for i in range(len(text)):
        if (i % 64 == 0):
            returned += "\n"
        if (i % 8 == 0):
            returned += " "
        returned += text[i]
    return returned[1:]

def encode(text):
    encoded = ""
    for char in text:
        encoded += char.encode("utf-8").hex()
    return bin(int(encoded, 16))[2:]

def sizeList(metalist):
    return (len(metalist), round(sum(len(row) for row in metalist) / len(metalist)))

def preprep(text):
    bintext = encode(text)
    padded = bintext + "1"
    while len(padded) % 448 != 0:
        padded += "0"
    padded += bin(len(text))[2:]
    while len(padded) % 512 != 0:
        padded += "0"
    return padded

def F(b, c, d, i):
    if 0 <= i <= 15:
        return  bOr(bAnd(b, c), bAnd(bNot(b), d))
    if 16 <= i <= 31:
        return bOr(bAnd(d, b), bAnd(bNot(d), c))
    if 32 <= i <= 47:
        return bXor(bXor(b, c), d)
    if 48 <= i <= 63:
        return bXor(c, bAnd(b, bNot(d)))

def combine(a, b, c, d, input, i):
    return binRot((F(b, c, d, i) + a + input) + K[i], r[i]) + b

In [123]:
chunky = [preprep("bleep")[i:i + 32] for i in range(0, len(preprep("bleep")), 32)]
sizeList(chunky)

(16, 32)

In [61]:
shiftrot = [[kval.strip() for kval in line.split(",")] for line in open("shiftrot.txt", "r").read().split("\n")]
K = concat(shiftrot[17:33])
print(K)

['11010111011010101010010001111000', '11101000110001111011011101010110', '00100100001000000111000011011011', '11000001101111011100111011101110', '11110101011111000000111110101111', '01000111100001111100011000101010', '10101000001100000100011000010011', '11111101010001101001010100000001', '01101001100000001001100011011000', '10001011010001001111011110101111', '11111111111111110101101110110001', '10001001010111001101011110111110', '01101011100100000001000100100010', '11111101100110000111000110010011', '10100110011110010100001110001110', '01001001101101000000100000100001', '11110110000111100010010101100010', '11000000010000001011001101000000', '00100110010111100101101001010001', '11101001101101101100011110101010', '11010110001011110001000001011101', '00000010010001000001010001010011', '11011000101000011110011010000001', '11100111110100111111101111001000', '00100001111000011100110111100110', '11000011001101110000011111010110', '11110100110101010000110110000111', '01000101010110100001010011

In [ ]:
binTestA =   "00110110111111000011110100100010"
binTestB =   "00001101111001000101000001101010"
bintTestA = 0b00110110111111000011110100100010
bintTestB = 0b00001101111001000101000001101010

In [20]:
# Tests of bitwise operations
print(bAnd(binTestA, binTestB) == binPadder(str(bin(bintTestA & bintTestB)[2:]), 32))
print(bOr(binTestA, binTestB) == binPadder(str(bin(bintTestA | bintTestB)[2:]), 32))
print(bXor(binTestA, binTestB) == binPadder(str(bin(bintTestA ^ bintTestB)[2:]), 32))
print(bNot(binTestA) != False) #Manually verified because the built-in bitwise NOT operator didn't work
print(bAdd(binTestA, binTestB) == binPadder(str(bin((bintTestA + bintTestB) % (2**32))[2:]), 32))
print(binRot(binTestA, 5) == binPadder(str(bin(((bintTestA << 5) | (bintTestA >> (32 - 5))) % (2**32))[2:]), 32))

True
True
True
True
True
True


In [124]:

def md5(plaintext):
    chunks = [preprep(plaintext)[i:i + 32] for i in range(0, len(preprep(plaintext)), 32)]
    if sizeList(chunks) != (16, 32):
        raise ValueError("Chunks got split wrong")
    for textChunk in chunks:
        ...
    ciphertext = ""
    return ciphertext

def sha1(plaintext):
    ciphertext = ""
    return ciphertext

def sha256(plaintext):
    ciphertext = ""
    return ciphertext